# Game Visualizer

Step through games from evaluation PGN files. Use the slider to move through positions.

In [1]:
import glob
import chess
import chess.pgn
import chess.svg
from IPython.display import display, SVG, HTML
import ipywidgets as widgets

In [2]:
# Load all games from the most recent PGN file (or set path manually)
pgn_files = sorted(glob.glob("results/games/*.pgn"))
if not pgn_files:
    raise FileNotFoundError("No PGN files found in results/games/. Run evaluate.py first.")

pgn_path = pgn_files[-1]  # most recent
print(f"Loading: {pgn_path}")

# pgn_path = 'results/games/eval_elo1350_20260319_190944.pgn'

games = []
with open(pgn_path) as f:
    while True:
        game = chess.pgn.read_game(f)
        if game is None:
            break
        games.append(game)

print(f"Loaded {len(games)} game(s)")

Loading: results/games/eval_elo1350_20260320_180059.pgn
Loaded 4 game(s)


In [ ]:
# Interactive game viewer — persistent widgets, no disappearing slider
board_output = widgets.Output()
move_label = widgets.HTML(value="Starting position")
slider = widgets.IntSlider(
    value=0, min=0, max=0,
    description="Move:", continuous_update=False,
    layout=widgets.Layout(width="400px"),
)
header = widgets.HTML()

# State shared between callbacks
current = {"positions": [], "moves": []}

def load_game(game_idx):
    game = games[game_idx]
    moves = list(game.mainline_moves())
    board = game.board()
    positions = [board.copy()]
    for move in moves:
        board.push(move)
        positions.append(board.copy())

    current["positions"] = positions
    current["moves"] = moves

    white = game.headers.get("White", "?")
    black = game.headers.get("Black", "?")
    result = game.headers.get("Result", "*")
    header.value = f"<b>Game {game_idx+1}:</b> {white} vs {black} — <b>{result}</b>"

    slider.max = len(moves)
    slider.value = 0
    show_position(0)

def show_position(move_num):
    moves = current["moves"]
    positions = current["positions"]
    last_move = moves[move_num - 1] if move_num > 0 else None
    svg = chess.svg.board(positions[move_num], lastmove=last_move, size=400)
    board_output.clear_output(wait=True)
    with board_output:
        display(SVG(svg))
    if move_num == 0:
        move_label.value = "Starting position"
    else:
        move_num_display = (move_num - 1) // 2 + 1
        dot = "." if (move_num - 1) % 2 == 0 else "..."
        move_label.value = f"Move {move_num_display}{dot} {moves[move_num-1]}"

slider.observe(lambda change: show_position(change["new"]), names="value")

# Game selector
game_selector = widgets.Dropdown(
    options=[(f"Game {i+1}: {g.headers.get('White','?')} vs {g.headers.get('Black','?')} [{g.headers.get('Result','*')}]", i)
             for i, g in enumerate(games)],
    description="Game:",
    layout=widgets.Layout(width="500px"),
)
game_selector.observe(lambda change: load_game(change["new"]), names="value")

display(game_selector)
display(header)
display(board_output)
display(slider)
display(move_label)

# Show first game
load_game(0)

Dropdown(description='Game:', layout=Layout(width='500px'), options=(('Game 1: AlphaZero vs Stockfish (ELO 135…

HTML(value='')

Output()

IntSlider(value=0, continuous_update=False, description='Move:', layout=Layout(width='400px'), max=0)

HTML(value='Starting position')